# Default assumed user expertise — figures

Run top to bottom. All figures save to `results/`.

| exp | anchors | file |
|---|---|---|
| 1 | style-written expert/novice prompts | `data/acts.npz` |
| 2 | addressed suffixes, behavioural readout | `results/behavior.jsonl` |
| 3 | addressed suffixes, activation readout | `results/probe2_positions.npz` |

In [ ]:
%load_ext autoreload
%autoreload 2
import json, numpy as np, matplotlib.pyplot as plt
from collections import defaultdict
from scipy.stats import spearmanr
from config import TOPICS, DATA, RESULTS
import analyze as A

plt.rcParams.update({'figure.dpi': 120, 'font.size': 9,
                     'axes.spines.top': False, 'axes.spines.right': False})
TOPICS_L = list(TOPICS)
NOV, EXP, BEH, GREY = '#c0603a', '#3a6ea5', '#5b8c5a', '#bbbbbb'
RESULTS.mkdir(exist_ok=True)
rng = np.random.default_rng(0)

def boot(x, n=4000):
    x = np.asarray(x)
    return np.percentile(rng.choice(x, (n, len(x)), replace=True).mean(1), [2.5, 97.5])

def barh(order, mu, ci, color, xlabel, title, fname, anchors=True):
    fig, ax = plt.subplots(figsize=(5.5, .34 * len(order) + 1.4))
    ax.barh(range(len(order)), mu, color=color, height=.68,
            xerr=np.abs(np.asarray(ci).T - mu), error_kw=dict(lw=1, ecolor='#333'))
    if anchors:
        ax.axvline(0, c=NOV, lw=1.5); ax.axvline(1, c=EXP, lw=1.5)
    ax.set_yticks(range(len(order))); ax.set_yticklabels(order)
    ax.set_xlabel(xlabel); ax.set_title(title, loc='left')
    plt.tight_layout(); plt.savefig(RESULTS / fname, bbox_inches='tight'); plt.show()

def scatter(x, y, labels, xlabel, ylabel, fname, diag=False, color=EXP):
    r = spearmanr(x, y)
    fig, ax = plt.subplots(figsize=(3.8, 3.6))
    if diag:
        lo, hi = min(min(x), min(y)) - .03, max(max(x), max(y)) + .03
        ax.plot([lo, hi], [lo, hi], ls=':', c='gray', lw=1)
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    else:
        z = np.polyfit(x, y, 1); xs = np.linspace(min(x), max(x), 20)
        ax.plot(xs, np.polyval(z, xs), ls='--', c='gray', lw=1)
    ax.scatter(x, y, s=34, color=color, zorder=3)
    for t, a_, b_ in zip(labels, x, y):
        ax.annotate(t, (a_, b_), fontsize=7, xytext=(4, 2), textcoords='offset points')
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.set_title(f'$\\rho$={r.statistic:.2f}, p={r.pvalue:.3f}', loc='left')
    plt.tight_layout(); plt.savefig(RESULTS / fname, bbox_inches='tight'); plt.show()
    return r

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Experiment 1 — style-written anchors

In [ ]:
acts1, topic1, label1, idx1 = A.load(DATA / 'acts.npz')
sweep1 = A.layer_sweep(acts1, topic1, label1, idx1, TOPICS_L)
auc1 = sweep1.mean(1)
np.savez(RESULTS / 'exp1_sweep.npz', sweep=sweep1)
L1 = int(auc1.argmax())
print(f'L1 AUC={auc1[0]:.3f}   peak L{L1}={auc1[L1]:.3f}')

X1 = acts1[:, L1, :]
p1 = {}
for t in TOPICS_L:
    m = topic1 == t
    d = A.direction(X1[m & (label1 == 'expert')], X1[m & (label1 == 'novice')])
    lo_ = (X1[m & (label1 == 'novice')] @ d).mean()
    hi_ = (X1[m & (label1 == 'expert')] @ d).mean()
    p1[t] = (X1[m & (label1 == 'neutral')] @ d - lo_) / (hi_ - lo_)

o1 = sorted(TOPICS_L, key=lambda t: p1[t].mean())
barh(o1, np.array([p1[t].mean() for t in o1]), [boot(p1[t]) for t in o1], NOV,
     'position (style-written anchors)',
     'Exp 1 — ordering from confounded anchors', 'fig1b_exp1_positions.png')

### The confound: bag-of-words matches the probe

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rows1 = [json.loads(l) for l in open(DATA / 'prompts.jsonl')]
anch = [r for r in rows1 if r['label'] != 'neutral']
ya = np.array([r['label'] == 'expert' for r in anch])

bow_x = []
for held in TOPICS_L:
    tr = np.array([r['topic'] != held for r in anch])
    V = TfidfVectorizer(min_df=2)
    clf = LogisticRegression(max_iter=1000).fit(
        V.fit_transform([r['text'] for i, r in enumerate(anch) if tr[i]]), ya[tr])
    pr = clf.predict_proba(V.transform([r['text'] for i, r in enumerate(anch) if not tr[i]]))[:, 1]
    bow_x.append(roc_auc_score(ya[~tr], pr))
bow_x = np.array(bow_x)
print(f'BoW cross-topic AUC: {bow_x.mean():.3f}  (probe peak {auc1[L1]:.3f}, L1 {auc1[0]:.3f})')

wl = np.array([len(r['text'].split()) for r in rows1])
for lab in ['expert', 'novice', 'neutral']:
    w = wl[label1 == lab]
    print(f'  {lab:8s} {w.mean():5.1f} ± {w.std():4.1f} words')

d_all = A.direction(X1[label1 == 'expert'], X1[label1 == 'novice'])
s_all = X1 @ d_all
mn = label1 == 'neutral'
print(f'  corr(score, length): all {np.corrcoef(s_all, wl)[0,1]:.3f}, '
      f'within neutral {np.corrcoef(s_all[mn], wl[mn])[0,1]:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['probe\n(layer 1)', 'probe\n(best layer)', 'bag-of-words'],
       [auc1[0], auc1[L1], bow_x.mean()], color=[GREY, NOV, GREY], width=.6)
ax.axhline(.5, ls=':', c='gray', lw=1)
ax.set_ylim(.4, 1.03); ax.set_ylabel('cross-topic anchor AUC')
ax.set_title('Exp 1 anchors are separable from surface text alone', loc='left')
plt.tight_layout(); plt.savefig(RESULTS / 'fig1c_bow_vs_probe.png', bbox_inches='tight'); plt.show()

## Experiment 3 — addressed anchors

In [ ]:
p2f = np.load(RESULTS / 'probe2_positions.npz')
auc2 = p2f['aucs']
posA = {t: p2f[f'A_{t}'] for t in TOPICS_L}
posB = {t: p2f[f'B_{t}'] for t in TOPICS_L}
both = {t: np.concatenate([posA[t], posB[t]]) for t in TOPICS_L}

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(auc1, lw=2, color=NOV, label='Exp 1: style-written anchors')
ax.plot(auc2, lw=2, color=EXP, label='Exp 3: addressed anchors')
ax.axhline(.5, ls=':', c='gray', lw=1)
ax.set_xlabel('layer'); ax.set_ylabel('held-out anchor AUC'); ax.set_ylim(.45, 1.03)
ax.legend(frameon=False, loc='lower right')
ax.set_title('Style-written anchors separate before any computation', loc='left')
plt.tight_layout(); plt.savefig(RESULTS / 'fig1_layer_sweep.png', bbox_inches='tight'); plt.show()
print(f'Exp3  L1={auc2[0]:.3f}  peak L{int(auc2.argmax())}={auc2.max():.3f}')

In [ ]:
o2 = sorted(TOPICS_L, key=lambda t: both[t].mean())
mu2 = np.array([both[t].mean() for t in o2])
barh(o2, mu2, [boot(both[t]) for t in o2], EXP,
     'position of the no-information default',
     'The default user reads as novice-leaning in every domain',
     'fig2_probe_positions.png')
for t in o2:
    c = boot(both[t])
    print(f'{t:14s}{both[t].mean():6.2f}  [{c[0]:.2f}, {c[1]:.2f}]')

In [ ]:
scatter([posA[t].mean() for t in TOPICS_L], [posB[t].mean() for t in TOPICS_L],
        TOPICS_L, 'position, paraphrase set A', 'set B',
        'fig3_paraphrase.png', diag=True);

## Experiment 2 — behavioural

In [ ]:
beh = [json.loads(l) for l in open(RESULTS / 'behavior.jsonl')]
FEATS = ['long_word_frac', 'mean_sent_len', 'hedges', 'defines', 'simplifies', 'bullets']
g = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
for r in beh:
    for f in FEATS:
        g[f][r['topic']][r['arm']].append(r[f])

print('anchor separation (expert - novice), mean over topics')
for f in FEATS:
    sep = [np.mean(g[f][t]['expert']) - np.mean(g[f][t]['novice']) for t in g[f]]
    sgn = np.mean([s > 0 for s in sep])
    print(f'  {f:16s} {np.mean(sep):8.3f}   consistent sign in {sgn*100:3.0f}% of topics')

In [ ]:
FEAT = 'long_word_frac'
bpos = {}
for t, arms in g[FEAT].items():
    lo_, hi_ = np.mean(arms['novice']), np.mean(arms['expert'])
    if abs(hi_ - lo_) > 1e-6:
        bpos[t] = (np.array(arms['neutral']) - lo_) / (hi_ - lo_)

bo = sorted(bpos, key=lambda t: bpos[t].mean())
barh(bo, np.array([bpos[t].mean() for t in bo]), [boot(bpos[t]) for t in bo], BEH,
     f'position of default — {FEAT}',
     'Exp 2 — behavioural measure, same stimuli', 'fig4_behaviour.png')
print('dropped (no anchor separation):', [t for t in TOPICS_L if t not in bpos])

## Convergence

In [ ]:
sh = [t for t in TOPICS_L if t in bpos]
scatter([both[t].mean() for t in sh], [bpos[t].mean() for t in sh], sh,
        'probe position (Exp 3)', 'behavioural position (Exp 2)',
        'fig5_convergence.png', color=BEH)

print('Exp1 vs Exp3 ordering:',
      spearmanr([p1[t].mean() for t in TOPICS_L], [both[t].mean() for t in TOPICS_L]))

In [ ]:
bow_neu = {  # paste from probe2.py stdout
    'fitness': -0.293, 'mortgage': -0.291, 'houseplants': -0.291, 'python': -0.290,
    'sourdough': -0.288, 'chemo': -0.288, 'carrepair': -0.288, 'ml': -0.288,
    'musictheory': -0.281, 'taxlaw': -0.281,
}
def mm(d):
    v = np.array([d[t] for t in TOPICS_L]); return (v - v.min()) / (v.max() - v.min())
pb, bb = mm({t: both[t].mean() for t in TOPICS_L}), mm(bow_neu)
o = np.argsort(pb); x = np.arange(len(TOPICS_L))

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.bar(x - .2, pb[o], .38, color=EXP, label='activation probe')
ax.bar(x + .2, bb[o], .38, color=GREY, label='bag-of-words')
ax.set_xticks(x); ax.set_xticklabels([TOPICS_L[i] for i in o], rotation=40, ha='right')
ax.set_ylabel('min-max scaled'); ax.legend(frameon=False)
ax.set_title('Lexical baseline carries no topic signal on neutral prompts', loc='left')
plt.tight_layout(); plt.savefig(RESULTS / 'fig6_bow_control.png', bbox_inches='tight'); plt.show()
print('BoW raw range:', round(max(bow_neu.values()) - min(bow_neu.values()), 4))

## Numbers for the write-up

In [ ]:
import csv
with open(RESULTS / 'summary.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['topic', 'exp1_pos', 'exp3_pos', 'exp3_lo', 'exp3_hi', 'exp2_pos'])
    for t in TOPICS_L:
        c = boot(both[t])
        w.writerow([t, round(p1[t].mean(), 3), round(both[t].mean(), 3),
                    round(c[0], 3), round(c[1], 3),
                    round(bpos[t].mean(), 3) if t in bpos else ''])
print(open(RESULTS / 'summary.csv').read())